In [1]:
import modules.data as d
import modules.model2 as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
from pathlib import Path

####

import pandas as pd
from typing import Literal

####

# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()


*** Device() ***
device = cuda:3



---

In [2]:
# get data
brca = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:3)
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (75939, 19)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:3)
y                (567, 6)         Tensor (cuda:3)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [120]:
from modules.train import DataModule, MultiTrainingModule
from datetime import datetime
from typing import Literal

In [ ]:
class Experiment():
    def __init__(self, data, generator, num_epochs:int, num_trials:int, batch_size:int=16, val_size:int=0.15, test_size:int=0.15):
        # inst vars
        self.num_epochs = num_epochs
        self.num_trials = num_trials

        # init data module
        self.data_module = DataModule(
            X=data.X,
            y=data.y,
            generator=generator,
            batch_size=batch_size,
            val_size=val_size,
            test_size=test_size
        )

        # init runs
        self.runs = {}

    def add_run(self, run_name:str, model_class, model_kwargs:dict, training_class, training_kwargs:dict):
        self.runs[run_name] = MultiTrainingModule(
            model_class=model_class,
            model_kwargs=model_kwargs,
            training_class=training_class,
            training_kwargs={
                'data_module':self.data_module,
                'num_epochs':self.num_epochs,
                **training_kwargs
            },
            num_trials=self.num_trials,
            comment=run_name,
            autorun=False
        )

    def run_experiment(self):
        # run expt
        for name, model in self.runs.items():
            print(f'\n{name}')
            model.run()

        # summarize results
        results = {f'{name}':model.summary for name, model in self.runs.items()}
        self.summary = pd.concat(results.values(), keys=results.keys(), names=['model','stat'])
        self.summary.columns.name = 'metric'

        # save to csv
        date = datetime.now().strftime("%Y-%m-%d")
        time = datetime.now().strftime("%Hh%Mm%Ss").lower()
        self.summary.to_csv(f'./{date}_{time}_experiment_summary.csv')

        return self.summary
    
    def splice(self, value:str, level:Literal['model','stat','metric']):
        # splice for model, stat
        if (level == 'model') or (level == 'stat'):
            return self.summary.xs(value, level=level)
        
        # splice for metric
        elif level == 'metric':
            return self.summary[value].unstack(level='stat')
        
    def sort(self, metric:str, stat:str='mean', ascending:bool=True):
        # sort summary by a metric
        sorted = self.splice(stat, 'stat').sort_values(metric, ascending=ascending).index
        summary_sorted = self.summary.loc[(sorted, slice(None))]
        
        return summary_sorted
    
    def sort_splice(self, metric:str, stat:str='mean', ascending:bool=True):
        # sort
        summary_sorted = self.sort(metric, stat, ascending)

        # splice
        return summary_sorted.xs(stat, level='stat')
        


        

---

In [105]:
from itertools import product


def generate_combinations(input_dict:dict, prefix:str):
    # generate combinations of kwargs
    keys = input_dict.keys()
    values_product = product(*input_dict.values())
    kwargs = [dict(zip(keys, values)) for values in values_product]

    # get run names from kwargs
    names = []
    for kwarg in kwargs:

        name_list = []

        for key, value in kwarg.items():
            if value == True:
                name_list.append(str(key))
            elif value != False:
                name_list.append(f'{value}')

        name = prefix + '_' + '_'.join(name_list)
        names.append(name)

    # get run kwargs
    run_kwargs = dict(zip(names, kwargs))
    return run_kwargs

    
gcn_opts = {
    'adj_in':(True, False),
    'adj_out':(True, False),
    'adj_self':(True, False),
    'bias':(True, False),
    'normalize':('row','sym'),
}

gcn_runs = generate_combinations(gcn_opts, 'GCN')
gcn_runs

{'GCN_adj_in_adj_out_adj_self_bias_row': {'adj_in': True,
  'adj_out': True,
  'adj_self': True,
  'bias': True,
  'normalize': 'row'},
 'GCN_adj_in_adj_out_adj_self_bias_sym': {'adj_in': True,
  'adj_out': True,
  'adj_self': True,
  'bias': True,
  'normalize': 'sym'},
 'GCN_adj_in_adj_out_adj_self_row': {'adj_in': True,
  'adj_out': True,
  'adj_self': True,
  'bias': False,
  'normalize': 'row'},
 'GCN_adj_in_adj_out_adj_self_sym': {'adj_in': True,
  'adj_out': True,
  'adj_self': True,
  'bias': False,
  'normalize': 'sym'},
 'GCN_adj_in_adj_out_bias_row': {'adj_in': True,
  'adj_out': True,
  'adj_self': False,
  'bias': True,
  'normalize': 'row'},
 'GCN_adj_in_adj_out_bias_sym': {'adj_in': True,
  'adj_out': True,
  'adj_self': False,
  'bias': True,
  'normalize': 'sym'},
 'GCN_adj_in_adj_out_row': {'adj_in': True,
  'adj_out': True,
  'adj_self': False,
  'bias': False,
  'normalize': 'row'},
 'GCN_adj_in_adj_out_sym': {'adj_in': True,
  'adj_out': True,
  'adj_self': False,


In [106]:
data = brca

exp = Experiment(
    data=data,
    generator=generator,
    num_epochs=30,
    num_trials=50,
    batch_size=32
)

exp.add_run(
    run_name='MLP',
    model_class=m.MLPClassifier,
    model_kwargs={
        'in_features':data.num_nodes,
        'out_features':data.num_labels,
        'flatten':True,
    },
    training_class=t.ClassifierTrainingModule,
    training_kwargs={
        'loss_fn':nn.CrossEntropyLoss(data.class_weights),
        'optimizer':torch.optim.Adam,
        'verbose':False
    },
)

for run_name, run_kwargs in gcn_runs.items():
    exp.add_run(
        run_name=run_name,
        model_class=m.GCNClassifier,
        model_kwargs={
            'in_features':data.num_features,
            'out_features':data.num_labels,
            'relation':data.relation,
            'num_nodes':data.num_nodes,
            'bias':False,
            'normalize':'row',
            **run_kwargs
        },
        training_class=t.ClassifierTrainingModule,
        training_kwargs={
            'loss_fn':nn.CrossEntropyLoss(data.class_weights),
            'optimizer':torch.optim.Adam,
            'verbose':False
        },
    )

exp.run_experiment()


MLP


100%|██████████| 50/50 [00:50<00:00,  1.02s/it]



GCN_adj_in_adj_out_adj_self_bias_row


100%|██████████| 50/50 [02:25<00:00,  2.92s/it]



GCN_adj_in_adj_out_adj_self_bias_sym


...


GCN_row


100%|██████████| 50/50 [00:49<00:00,  1.00it/s]



GCN_sym


100%|██████████| 50/50 [00:49<00:00,  1.01it/s]


metric                                     tot_loss      accuracy  \
model                                stat                           
MLP                                  mean  2.600487  8.955294e-01   
                                     std   0.228929  1.438718e-02   
GCN_adj_in_adj_out_adj_self_bias_row mean  3.054276  8.854118e-01   
                                     std   0.374617  1.838167e-02   
GCN_adj_in_adj_out_adj_self_bias_sym mean  3.046341  8.865882e-01   
...                                             ...           ...   
GCN_bias_sym                         std   0.027651  8.040186e-02   
GCN_row                              mean  9.910407  2.352941e-01   
                                     std   0.000000  2.803737e-16   
GCN_sym                              mean  9.910407  2.352941e-01   
                                     std   0.000000  2.803737e-16   

metric                                        precision        recall  \
model                                stat                               
MLP                                  mean  8.907186e-01  8.955294e-01   
                                     std   1.737522e-02  1.438718e-02   
GCN_adj_in_adj_out_adj_self_bias_row mean  8.809577e-01  8.854118e-01   
                                     std   2.207465e-02  1.838167e-02   
GCN_adj_in_adj_out_adj_self_bias_sym mean  8.848523e-01  8.865882e-01   
...                                                 ...           ...   
GCN_bias_sym                         std   2.558288e-02  8.040186e-02   
GCN_row                              mean  5.536332e-02  2.352941e-01   
                                     std   3.504671e-17  2.803737e-16   
GCN_sym                              mean  5.536332e-02  2.352941e-01   
                                     std   3.504671e-17  2.803737e-16   

metric                                           f1  
model                                stat            
MLP                                  mean  0.890753  
                                     std   0.014774  
GCN_adj_in_adj_out_adj_self_bias_row mean  0.879054  
                                     std   0.020906  
GCN_adj_in_adj_out_adj_self_bias_sym mean  0.880650  
...                                             ...  
GCN_bias_sym                         std   0.039523  
GCN_row                              mean  0.089636  
                                     std   0.000000  
GCN_sym                              mean  0.089636  
                                     std   0.000000  

[66 rows x 5 columns]

In [118]:
sorted = exp.splice('mean', 'stat').sort_values('tot_loss', ascending=True).index
summary_sorted = exp.summary.loc[(sorted, slice(None))]
summary_sorted

metric                     tot_loss      accuracy     precision        recall  \
model                stat                                                       
MLP                  mean  2.600487  8.955294e-01  8.907186e-01  8.955294e-01   
                     std   0.228929  1.438718e-02  1.737522e-02  1.438718e-02   
GCN_adj_in_row       mean  2.656577  8.663529e-01  8.648667e-01  8.663529e-01   
                     std   0.409550  1.912720e-02  2.347628e-02  1.912720e-02   
GCN_adj_out_bias_row mean  2.724354  8.837647e-01  8.906643e-01  8.837647e-01   
...                             ...           ...           ...           ...   
GCN_sym              std   0.000000  2.803737e-16  3.504671e-17  2.803737e-16   
GCN_bias_sym         mean  9.936287  1.437647e-01  2.700346e-02  1.437647e-01   
                     std   0.027651  8.040186e-02  2.558288e-02  8.040186e-02   
GCN_bias_row         mean  9.939611  1.609412e-01  3.217716e-02  1.609412e-01   
                     std   0.023334  8.001977e-02  2.416725e-02  8.001977e-02   

metric                           f1  
model                stat            
MLP                  mean  0.890753  
                     std   0.014774  
GCN_adj_in_row       mean  0.856146  
                     std   0.023674  
GCN_adj_out_bias_row mean  0.881240  
...                             ...  
GCN_sym              std   0.000000  
GCN_bias_sym         mean  0.044593  
                     std   0.039523  
GCN_bias_row         mean  0.052937  
                     std   0.037957  

[66 rows x 5 columns]

In [122]:
summary_sorted.xs('mean', level='stat')

metric,tot_loss,accuracy,precision,recall,f1
model,,,,,
MLP,2.600487,0.895529,0.890719,0.895529,0.890753
GCN_adj_in_row,2.656577,0.866353,0.864867,0.866353,0.856146
GCN_adj_out_bias_row,2.724354,0.883765,0.890664,0.883765,0.881240
GCN_adj_self_sym,2.725229,0.899294,0.895358,0.899294,0.894943
GCN_adj_in_bias_sym,2.725756,0.875529,0.878517,0.875529,0.870625
GCN_adj_in_bias_row,2.737076,0.877176,0.875258,0.877176,0.868756
GCN_adj_self_bias_sym,2.749655,0.896706,0.893226,0.896706,0.892128
GCN_adj_self_row,2.759775,0.899059,0.895195,0.899059,0.894570
GCN_adj_in_sym,2.767101,0.870118,0.867479,0.870118,0.863959


In [121]:
exp.splice('mean', 'stat')

metric,tot_loss,accuracy,precision,recall,f1
model,,,,,
MLP,2.600487,0.895529,0.890719,0.895529,0.890753
GCN_adj_in_adj_out_adj_self_bias_row,3.054276,0.885412,0.880958,0.885412,0.879054
GCN_adj_in_adj_out_adj_self_bias_sym,3.046341,0.886588,0.884852,0.886588,0.880650
GCN_adj_in_adj_out_adj_self_row,3.135399,0.887294,0.883604,0.887294,0.881669
GCN_adj_in_adj_out_adj_self_sym,3.062820,0.880471,0.879948,0.880471,0.873711
GCN_adj_in_adj_out_bias_row,2.939970,0.869882,0.870582,0.869882,0.862894
GCN_adj_in_adj_out_bias_sym,3.016841,0.875294,0.878216,0.875294,0.869770
GCN_adj_in_adj_out_row,2.799698,0.863294,0.863882,0.863294,0.855071
GCN_adj_in_adj_out_sym,3.049247,0.868235,0.871373,0.868235,0.862062


In [112]:
exp.summary.to_csv('./gcn_adj_ablation.csv')

In [119]:
summary_sorted.to_csv('./gcn_adj_ablation_loss_sorted.csv')

In [117]:
Path('./output') / '' / 'hello.csv'

PosixPath('output/hello.csv')